In [1]:
PROJECT_ID = "qwiklabs-gcp-01-5fc8b912fc6a"
DATASET    = "fraud_detection"      # reuse your dataset, or make a new one
LOCATION   = "US"
GCS_URI    = "gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv"

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

In [2]:
raw_table = f"{PROJECT_ID}.{DATASET}.emergency_calls"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)
client.load_table_from_uri(GCS_URI, raw_table, job_config=job_config).result()

print("Loaded rows:", client.get_table(raw_table).num_rows)

Loaded rows: 50000


In [9]:
# look at the columns so we know the response-time column and the features
client.query(f"SELECT * FROM `{raw_table}` LIMIT 5").to_dataframe()

,call_id,call_timestamp,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available,response_time
0,35957,2023-01-01 00:05:53+00:00,Fire,Highland,Rainy,Sunday,0,High,21.45,3,23.41
1,20832,2023-01-01 00:20:47+00:00,Fire,Oakmont,Rainy,Sunday,0,High,22.29,6,20.11
2,27949,2023-01-01 00:33:27+00:00,Fire,Riverside,Windy,Sunday,0,High,17.19,14,19.75
3,20199,2023-01-01 00:48:29+00:00,Fire,Riverside,Windy,Sunday,0,High,17.39,14,20.76
4,46938,2023-01-01 00:50:44+00:00,Rescue,Brookfield,Sunny,Sunday,0,High,22.50,14,22.37


In [11]:
model = f"{PROJECT_ID}.{DATASET}.response_time_model"

client.query(f"""
CREATE OR REPLACE MODEL `{model}`
OPTIONS (model_type='linear_reg', input_label_cols=['response_time']) AS
SELECT * EXCEPT(call_id, call_timestamp) FROM `{raw_table}`
WHERE response_time IS NOT NULL
""").result()

print("model trained")

model trained


In [12]:
client.query(f"SELECT * FROM ML.EVALUATE(MODEL `{model}`)").to_dataframe()

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,1.742121,4.770664,0.014883,1.474354,0.829923,0.829955


In [13]:
# feed in one made-up call and see the predicted response time
client.query(f"""
SELECT * FROM ML.PREDICT(MODEL `{model}`, (
  SELECT
    'Fire'      AS call_type,
    'Riverside' AS location,
    'Rainy'     AS weather_condition,
    'Sunday'    AS day_of_week,
    18          AS time_of_day,
    'High'      AS traffic_level,
    20.0        AS distance_to_station,
    5           AS units_available
))
""").to_dataframe()

,predicted_response_time,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available
0,21.488295,Fire,Riverside,Rainy,Sunday,18,High,20.0,5
